# PSF Defocus Z-Scan Analysis

Vectorial Debye PSF simulation sweep for **ATTO 488**, **ATTO 565**, and **ATTO 647N**.  
Quantifies the effect of axial defocus on localisation precision and colour precision  
using the standard S3M fitting pipeline (`FittingStrategy.STANDARD`).

Physics model: vectorial Debye + Gibson–Lanni spherical aberration (NA 1.49, oil/water).

**Structure**
1. PSF shape visualisation — per-channel (B/G/R) RGB composite at increasing defocus  
2. Z-sweep simulation — 7 z-offsets × 2 coverslip depths × 3 photon levels  
3. Analysis — localisation precision and colour precision vs defocus

In [ ]:
import sys, os, glob, types
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import polars as pl

sys.path.append(str(Path(os.getcwd()).parent.parent))

from src import IOFunctions, SpectralFunctions, MaskFunctions, sCMOSFunctions, PlottingBase
from src.Multicolour_Simulation_Functions import (
    FittingStrategy, SimulationConfig, MultiC_Sim_Funcs,
)
from src.simulation.defocus_psf import VectorialPSF

IO    = IOFunctions.IO_Functions()
S_F   = SpectralFunctions.Spectral_Funcs()
M_F   = MaskFunctions.Mask_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
MSF   = MultiC_Sim_Funcs()
plotter = PlottingBase.PublicationPlotter()

In [ ]:
# ── Camera calibration ────────────────────────────────────────────────────────
cal_folder = Path("../../Camera_Calibrations/Ximea_Camera/")
gain      = IO.read_tiff(str(cal_folder / "gain.tif"))
offset    = IO.read_tiff(str(cal_folder / "offset.tif"))
variance  = IO.read_tiff(str(cal_folder / "variance.tif"))
readnoise = float(np.median(IO.read_tiff(str(cal_folder / "readnoise.tif"))))
rqe       = IO.read_tiff(str(cal_folder / "rqe.tif"))

print(f"Camera calibration loaded — sensor: {gain.shape}, readnoise: {readnoise:.2f}")

In [ ]:
# ── Spectral setup ────────────────────────────────────────────────────────────
R, G, B, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B, G, R])   # (3, n_wl); convention B=0, G=1, R=2

# Dyes and global simulation constants
dyes          = ["ATTO 488", "ATTO 565", "ATTO 647N"]
filters       = []          # no bandpass filter
NA            = 1.49
pixel_size_nm = 69.0        # Ximea object-space pixel (nm)
image_dims    = 32          # 22 × 22 px  ≈  1.52 µm FOV
background_photons = 40.0

# Smoothing function (Gaussian, σ = 1.5 px) ─ must be defined before MSF calls
smoothing_function = types.SimpleNamespace(
    args={"sigma": 1.5},
    extent=1.5,
    smoothing_function=sCMOS.gaussian_filter_stack,
    data_arg="image",
)

# Coarser wavelength grid for PSF integration (5 nm steps; sufficient to <1% PSF error)
wl_psf_nm = np.arange(400, 751, 5, dtype=float)

# build_spectral_weights requires pixel_QYs on the same grid as wl_psf_nm;
# the native grid from getpixelefficiency() may differ — interpolate.
pixel_QYs_psf = np.vstack([
    np.interp(wl_psf_nm, wavelength, pixel_QYs[c])
    for c in range(pixel_QYs.shape[0])
])  # (3, len(wl_psf_nm))

print(f"Wavelength grid: {wavelength[0]:.0f}–{wavelength[-1]:.0f} nm  "
      f"({len(wavelength)} pts);  PSF grid: {len(wl_psf_nm)} pts @ 5 nm steps")

In [ ]:
# ── Save folder ───────────────────────────────────────────────────────────────
save_folder = Path("/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/Simulation/Z_Position_Effect")
save_folder.mkdir(parents=True, exist_ok=True)
print(f"Results → {save_folder}")

In [ ]:
# ── VectorialPSF objects and per-dye spectral weights ─────────────────────────
# N_pupil=128 is fast enough for the display figure; simulation dispatch uses 256
vpsf_display = VectorialPSF(
    NA=NA, n_medium=1.33, n_immersion=1.515,
    pix_obj_um=pixel_size_nm / 1000.0,
    psf_size=21, N_pupil=256,
)

# Spectral weights: w_c(λ) = S(λ)·T_obj(λ)·QE_c(λ), shape (n_wl_psf, 3) = (n_wl_psf, B/G/R)
# pixel_QYs_psf is already interpolated to wl_psf_nm in the spectral setup cell.
spectral_weights = {}
channel_fractions = {}
for dye in dyes:
    w = VectorialPSF.build_spectral_weights(
        spectral_functions=S_F,
        dye=dye,
        filters=None,
        wavelengths_nm=wl_psf_nm,
        pixel_QYs=pixel_QYs_psf,   # (3, n_wl_psf) — matched grid
        include_objective=True,
    )  # (n_wl_psf, 3)
    spectral_weights[dye] = w
    frac = w.sum(axis=0) / w.sum()
    channel_fractions[dye] = frac
    print(f"{dye:12s}  B={frac[0]:.3f}  G={frac[1]:.3f}  R={frac[2]:.3f}")

## PSF Shape Visualisation

Polychromatic vectorial PSF at each defocus step, integrated over the dye emission  
spectrum weighted by per-channel Bayer QE.  Each panel is an **RGB composite**:  
B-channel PSF → blue, G-channel PSF → green, R-channel PSF → red.  
Intensities are normalised to the peak of the **in-focus** PSF for that channel,  
so dimming of the central peak with defocus is visible.

In [ ]:
# ── Compute PSF patches for visualisation ────────────────────────────────────
z_display_nm = np.array([0, 100, 200, 300, 500], dtype=float)
z_display_um = z_display_nm / 1000.0
wl_display_um = wl_psf_nm / 1000.0

psf_display = {}    # psf_display[dye] shape: (n_z, 3, psf_size, psf_size)
for dye in dyes:
    psf_display[dye] = vpsf_display.compute_psf_stack(
        z_offsets_um=z_display_um,
        wavelengths_um=wl_display_um,
        spectral_weights=spectral_weights[dye],
        distance_from_coverslip_um=0.0,
    )  # (n_z, 3, 21, 21)
    print(f"{dye} PSF stack: {psf_display[dye].shape}  "
          f"in-focus B/G/R peak: {psf_display[dye][0].max(axis=(-2,-1))}")

In [ ]:
# ── Figure 1: raw vectorial PSF at increasing defocus ────────────────────────
# |FFT(E)|² from the Debye model: no noise, no camera QE, no Bayer mask.
# 25 nm/pixel oversamples the Airy disk (~9 px across) to show ring structure.
# Each panel normalised to its own peak to reveal structure at every defocus.

# pix_raw_nm   = 5.0
# psf_size_raw = int(3000/5)          # 81 × 25 nm = 2.025 µm window

# vpsf_raw = VectorialPSF(
#     NA=NA, n_medium=1.33, n_immersion=1.515,
#     pix_obj_um=pix_raw_nm / 1000.0,
#     psf_size=psf_size_raw,
#     N_pupil=512,
# )

# z_disp_raw_nm = np.array([0, 100, 250, 500], dtype=float)

# psf_raw_stack = vpsf_raw.compute_psf_stack(
#     z_offsets_um=z_disp_raw_nm / 1000.0,
#     wavelengths_um=np.array([0.55]),
#     spectral_weights=None,
#     distance_from_coverslip_um=0.0,
# )  # (n_z, 1, ps, ps)

# n_z_raw = len(z_disp_raw_nm)

# fig, axs = plotter.two_column_plot(nrows=1, ncols=n_z_raw, height=6.69 / n_z_raw + 0.05)
# fs = plotter.config.font_size

# for i_z in range(n_z_raw):
#     I = psf_raw_stack[i_z, 0]
#     I_disp = I / I.max()
#     I_log = np.log10(np.maximum(I_disp, 1e-4))   # floor avoids log(0)
#     plotter.image_plot(
#         axs[i_z], I_log,
#         cmap="gray",
#         vmin=np.percentile(I_log, 0.1), vmax=np.percentile(I_log, 99.9),
#         scalebar=True,
#         pixelsize=pix_raw_nm,
#         scalebarsize=500.0,
#         scalebarlabel="500 nm",
#         scalebar_color="white",
#         origin="lower",
#         colorbar=(i_z == 3)
#     )

#     # In-panel annotation: defocus label in top-left corner
#     label = f"Δz = {z_disp_raw_nm[i_z]:.0f} nm"
#     axs[i_z].text(
#         0.05, 0.97, label,
#         transform=axs[i_z].transAxes,
#         fontsize=fs, color="white",
#         va="top", ha="left",
#         bbox=dict(boxstyle="square,pad=0.15", fc="none", ec="none"),
#     )

# plotter.save_or_show(fig, save_path=str(save_folder / "psf_defocus_raw.svg"), show=False)

## Z-Sweep Simulation

For each (dye, z-offset, coverslip depth), run `test_simulation_method` with  
`FittingStrategy.STANDARD` across three photon levels.  
Two coverslip depths probe the effect of depth-induced spherical aberration:
- **0 nm** — emitter at the coverslip (no spherical aberration)  
- **500 nm** — emitter 500 nm into the aqueous sample (realistic cell imaging)

In [ ]:
# ── Sweep parameters ─────────────────────────────────────────────────────────
z_offsets_nm       = np.hstack([0, np.geomspace(5, 500, 49)])
coverslip_depths_nm = np.array([0, 500], dtype=float)
n_photon_space     = np.geomspace(500, 20000, 100)

# n_bootstrap: number of simulated localisations per (z, photon, dye, d_coverslip).
# 2000 gives ~3% RMSE uncertainty (1/√2000); increase to 10 000 for publication quality.
n_bootstrap = 10000

print(f"Sweep:  {len(dyes)} dyes  ×  {len(z_offsets_nm)} z-offsets  "
      f"×  {len(coverslip_depths_nm)} coverslip depths  "
      f"×  {len(n_photon_space)} photon levels")
print(f"Total simulation calls: "
      f"{len(dyes) * len(z_offsets_nm) * len(coverslip_depths_nm)}")

In [ ]:
# ── Camera parameters dictionary (constant across sweep) ─────────────────────
def make_camera_params(image_dims):
    """Median-flat camera calibration for a square image of size image_dims."""
    return {
        "gain":               np.full((image_dims, image_dims), np.median(gain)),
        "offset":             np.full((image_dims, image_dims), np.median(offset)),
        "variance":           np.full((image_dims, image_dims), np.median(variance)),
        "readnoise":          readnoise,
        "rqe":                np.full((image_dims, image_dims), np.median(rqe)),
        "masks":              M_F.get_masks(size_x=image_dims, size_y=image_dims),
        "pixel_QYs":          pixel_QYs,
        "pixel_order":        ["B", "G", "R"],
        "pixel_order_indices": {"B": 0, "G": 1, "R": 2},
    }

camera_params_dict = make_camera_params(image_dims)
print(f"Camera parameters dict ready — image size: {image_dims}×{image_dims} px")

In [ ]:
# ── Run sweep ─────────────────────────────────────────────────────────────────
import time

def _flag_nm(val: float) -> str:
    """Format a nm value for use in a filename flag.

    Uses 1 decimal place with 'p' as the decimal separator (e.g. 5.5 → '00005p5'),
    zero-padded to 7 characters so filenames sort correctly.  1 d.p. is sufficient
    to keep all geomspace(5, 500, 49) values unique (minimum step ≈ 0.5 nm at z=5 nm).
    """
    return f"{np.around(val, 1):.1f}".replace(".", "p").zfill(7)

n_total  = len(dyes) * len(z_offsets_nm) * len(coverslip_depths_nm)
i_run    = 0
t_start  = time.time()

for dye in dyes:
    for z_nm in z_offsets_nm:
        for d_nm in coverslip_depths_nm:
            i_run += 1
            flag = f"defocus_z{_flag_nm(z_nm)}nm_d{_flag_nm(d_nm)}nm_"

            config = SimulationConfig(
                n_bootstrap=n_bootstrap,
                background_photons=background_photons,
                NA=NA,
                pixel_size=pixel_size_nm,
                save_raw_results=True,
                subtractx0y0=False,
                use_stochastic_photons=True,
                save_summary_csvs=True,
                verbose=False,
                defocus_z_um=z_nm / 1000.0,
                distance_from_coverslip_um=d_nm / 1000.0,
            )

            print(f"[{i_run:3d}/{n_total}]  {dye}  z={z_nm:6.1f} nm  "
                  f"d={d_nm:4.0f} nm", end="  ", flush=True)

            MSF.test_simulation_method(
                dye=dye,
                filters=filters,
                wavelength=wavelength,
                camera_parameters=camera_params_dict,
                save_folder=str(save_folder),
                n_photon_space=n_photon_space,
                smoothing_function=smoothing_function,
                strategy=FittingStrategy.STANDARD,
                starting_flag=flag,
                config=config,
                overwrite=True,
            )

            elapsed = (time.time() - t_start) / 60.0
            rate    = i_run / elapsed if elapsed > 0 else 0
            eta     = (n_total - i_run) / rate if rate > 0 else 0
            print(f"done   ({elapsed:.1f} min elapsed, ETA {eta:.1f} min)")

print(f"\nSweep complete in {(time.time()-t_start)/60:.1f} min.")

## Analysis

Load the RMSE summary CSVs saved by `test_simulation_method` and aggregate into  
an xarray for plotting.  

Metrics (from `_compute_fit_statistics`):
- **`xc`, `yc`** — RMSE of fitted position in nm  
- **`colour_distance`** — mean Euclidean distance of fitted (A_B, A_G, A_R) from truth

In [ ]:
import xarray as xr
import matplotlib.colors as mcolors
import re

# ── Build xarray DataArray from RMSE summary CSVs ─────────────────────────────
# Dims: [dye, z_nm, coverslip_nm, n_photons, metric]
metrics = ["sigma_xy", "colour_dist"]

da_defocus = xr.DataArray(
    data=np.full(
        [len(dyes), len(z_offsets_nm), len(coverslip_depths_nm),
         len(n_photon_space), len(metrics)],
        np.nan,
    ),
    coords={
        "dye":          dyes,
        "z_nm":         z_offsets_nm,
        "coverslip_nm": coverslip_depths_nm,
        "n_photons":    n_photon_space,
        "metric":       metrics,
    },
    dims=["dye", "z_nm", "coverslip_nm", "n_photons", "metric"],
)

# Regex: capture the z and d values however they were formatted (integers, decimals,
# or 'p'-separated decimals like "5p0" or "10p50").
_flag_re = re.compile(
    r"defocus_z([\d]+(?:[p.][\d]+)?)nm_d([\d]+(?:[p.][\d]+)?)nm_",
    re.IGNORECASE,
)

def _parse_nm(s: str) -> float:
    """Convert a flag component like '0005', '5p0', or '5.5' to float nm."""
    return float(s.replace("p", "."))

def _nearest_idx(arr, val, tol=None):
    """Return index of nearest element in arr; raise if gap > tol (if given)."""
    idx = int(np.argmin(np.abs(arr - val)))
    if tol is not None and abs(arr[idx] - val) > tol:
        return None
    return idx

n_loaded = 0
missing  = []

for i_dye, dye in enumerate(dyes):
    dye_str = dye.replace("/", "-")
    # Scan ALL RMSE_mean files for this dye — don't assume the flag format
    all_files = sorted(
        glob.glob(str(save_folder / f"defocus_z*nm_d*nm_*{dye_str}*RMSE_mean*.csv"))
    )
    if not all_files:
        missing.append((dye, "all"))
        continue

    for fpath in all_files:
        fname = Path(fpath).name
        m = _flag_re.search(fname)
        if not m:
            continue
        z_val = _parse_nm(m.group(1))
        d_val = _parse_nm(m.group(2))

        i_z = _nearest_idx(z_offsets_nm,       z_val, tol=1.0)
        i_d = _nearest_idx(coverslip_depths_nm, d_val, tol=10.0)
        if i_z is None or i_d is None:
            continue  # outside the grid

        df = pl.read_csv(fpath)
        for i_ph, row in enumerate(df.iter_rows(named=True)):
            if i_ph >= len(n_photon_space):
                break
            da_defocus[i_dye, i_z, i_d, i_ph, 0] = float(
                np.sqrt((row["xc"] ** 2 + row["yc"] ** 2) / 2)
            )
            da_defocus[i_dye, i_z, i_d, i_ph, 1] = float(row["colour_distance"])
        n_loaded += 1

if missing:
    print(f"No files found for: {missing}")
n_filled = int(np.sum(~np.isnan(da_defocus.sel(metric="sigma_xy").values)))
print(f"Loaded {n_loaded} CSV files → {n_filled} grid cells filled "
      f"({n_filled / da_defocus.sel(metric='sigma_xy').size * 100:.1f} %)")
da_defocus.to_netcdf(str(save_folder / "defocus_sweep.nc"))
print(f"σ_xy  range : {float(da_defocus.sel(metric='sigma_xy').min()):.1f}–"
      f"{float(da_defocus.sel(metric='sigma_xy').max()):.1f} nm")
print(f"colour range: {float(da_defocus.sel(metric='colour_dist').min()):.4f}–"
      f"{float(da_defocus.sel(metric='colour_dist').max()):.4f}")

In [ ]:
# ── Plotting helpers ──────────────────────────────────────────────────────────
dye_labels   = {"ATTO 488": "ATTO 488", "ATTO 565": "ATTO 565", "ATTO 647N": "ATTO 647N"}
depth_labels = {0.0: "d = 0 nm (coverslip)", 500.0: "d = 500 nm (into sample)"}
fs = plotter.config.font_size

In [ ]:
# ── Figure 2: localisation precision σ_xy — surface map ──────────────────────
# X = defocus (nm), Y = photon count (log scale), colour = σ_xy (nm, viridis).
# Layout: 2 rows (coverslip depth) × 3 cols (dye); shared colour scale per figure.
metric     = "sigma_xy"
cbar_label = r"$\sigma_{xy}$ / nm"

data_m = da_defocus.sel(metric=metric).values  # (n_dye, n_z, n_d, n_ph)
vmin, vmax = float(np.nanmin(data_m)), float(np.nanmax(data_m))
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

fig, axs = plt.subplots(
    nrows=len(coverslip_depths_nm), ncols=len(dyes),
    figsize=(6.69, 2.5 * len(coverslip_depths_nm)),
    squeeze=False,
)
for i_d, d_nm in enumerate(coverslip_depths_nm):
    for i_c, dye in enumerate(dyes):
        ax = axs[i_d, i_c]
        # da shape at this point: (n_z, n_photons); transpose → (n_photons, n_z) for pcolormesh
        Z = da_defocus.isel(dye=i_c, coverslip_nm=i_d).sel(metric=metric).values
        ax.pcolormesh(z_offsets_nm, n_photon_space, Z.T,
                      cmap="viridis", shading="auto", norm=norm)
        ax.set_yscale("log")
        ax.set_xlabel("Defocus / nm", fontsize=fs)
        ax.set_ylabel(
            f"Photons\n{depth_labels[d_nm]}" if i_c == 0 else "",
            fontsize=fs - 1,
        )
        if i_d == 0:
            ax.set_title(dye_labels[dye], fontsize=fs)
        ax.tick_params(labelsize=fs - 1)

fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap="viridis"),
    ax=axs, label=cbar_label, shrink=0.7, aspect=30, pad=0.02,
)
plotter.save_or_show(fig, save_path=str(save_folder / "localisation_precision_surface.svg"))

In [ ]:
# ── Figure 3: colour precision — surface map ──────────────────────────────────
# Same layout: 2 rows × 3 cols. Shared colour scale across all dyes and depths.
metric     = "colour_dist"
cbar_label = r"$\sigma_{\mathrm{colour}}$ (a.u.)"

data_m = da_defocus.sel(metric=metric).values
vmin, vmax = float(np.nanmin(data_m)), float(np.nanmax(data_m))
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

fig, axs = plt.subplots(
    nrows=len(coverslip_depths_nm), ncols=len(dyes),
    figsize=(6.69, 2.5 * len(coverslip_depths_nm)),
    squeeze=False,
)
for i_d, d_nm in enumerate(coverslip_depths_nm):
    for i_c, dye in enumerate(dyes):
        ax = axs[i_d, i_c]
        Z = da_defocus.isel(dye=i_c, coverslip_nm=i_d).sel(metric=metric).values
        ax.pcolormesh(z_offsets_nm, n_photon_space, Z.T,
                      cmap="viridis", shading="auto", norm=norm)
        ax.set_yscale("log")
        ax.set_xlabel("Defocus / nm", fontsize=fs)
        ax.set_ylabel(
            f"Photons\n{depth_labels[d_nm]}" if i_c == 0 else "",
            fontsize=fs - 1,
        )
        if i_d == 0:
            ax.set_title(dye_labels[dye], fontsize=fs)
        ax.tick_params(labelsize=fs - 1)

fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap="viridis"),
    ax=axs, label=cbar_label, shrink=0.7, aspect=30, pad=0.02,
)
plotter.save_or_show(fig, save_path=str(save_folder / "colour_precision_surface.svg"))

In [ ]:
# ── Figure 4: combined summary — 2 metrics × 3 dyes, independent per-row scale ─
# Each row uses its own colour normalisation so both metrics are maximally readable.
# Coverslip depth = 0 nm (no spherical aberration) for the primary summary view.
metric_list  = ["sigma_xy",            "colour_dist"]
cbar_labels4 = [r"$\sigma_{xy}$ / nm", r"$\sigma_{\mathrm{colour}}$ (a.u.)"]
i_d_summary  = 0   # coverslip_nm index 0 → d = 0 nm

fig, axs = plt.subplots(
    nrows=len(metric_list), ncols=len(dyes),
    figsize=(6.69, 2.5 * len(metric_list)),
    squeeze=False,
)
for i_m, (metric, cbar_lbl) in enumerate(zip(metric_list, cbar_labels4)):
    # Normalise across all dyes at this coverslip depth
    data_row = da_defocus.isel(coverslip_nm=i_d_summary).sel(metric=metric).values
    vmin, vmax = float(np.nanmin(data_row)), float(np.nanmax(data_row))
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for i_c, dye in enumerate(dyes):
        ax = axs[i_m, i_c]
        Z = da_defocus.isel(dye=i_c, coverslip_nm=i_d_summary).sel(metric=metric).values
        ax.pcolormesh(z_offsets_nm, n_photon_space, Z.T,
                      cmap="viridis", shading="auto", norm=norm)
        ax.set_yscale("log")
        ax.set_xlabel("Defocus / nm", fontsize=fs)
        ax.set_ylabel("Photons" if i_c == 0 else "", fontsize=fs - 1)
        if i_m == 0:
            ax.set_title(dye_labels[dye], fontsize=fs)
        ax.tick_params(labelsize=fs - 1)

    fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap="viridis"),
        ax=axs[i_m, :].tolist(),
        label=cbar_lbl, shrink=0.85, aspect=20, pad=0.02,
    )

fig.suptitle(f"d = 0 nm (at coverslip)", fontsize=fs, y=1.01)
plotter.save_or_show(fig, save_path=str(save_folder / "combined_summary_surface.svg"))

## Bias Analysis

Bias = systematic offset of the **mean** fit from truth (distinct from precision/RMSE above).

- **Position bias** `bias_xy`: $\sqrt{(\bar\epsilon_x^2 + \bar\epsilon_y^2)/2}$ in nm, where $\bar\epsilon = \text{mean}(x_\text{fit} - x_0)$ across bootstraps.
- **Colour bias** `colour_bias`: Euclidean distance of the mean normalised fitted colour $\bar f_\text{fit} = \overline{A_\text{BGR}/\sum A}$ from the true BGR fraction $f_\text{true}$.

Both metrics use only valid localisations (positive $\sigma$, within image bounds).

In [ ]:
import pandas as pd
from scipy.spatial.distance import cdist

# ── Build bias DataArray from raw HDF5 + groundtruth/input_params CSVs ────────
# bias_xy     : sqrt((mean_ex^2 + mean_ey^2) / 2)  in nm
# colour_bias : ||mean(A_BGR / sum) − true_BGR||   (normalised fraction space)
metrics_bias = ["bias_xy", "colour_bias"]

da_defocus_bias = xr.DataArray(
    data=np.full(
        [len(dyes), len(z_offsets_nm), len(coverslip_depths_nm),
         len(n_photon_space), len(metrics_bias)],
        np.nan,
    ),
    coords={
        "dye":          dyes,
        "z_nm":         z_offsets_nm,
        "coverslip_nm": coverslip_depths_nm,
        "n_photons":    n_photon_space,
        "metric":       metrics_bias,
    },
    dims=["dye", "z_nm", "coverslip_nm", "n_photons", "metric"],
)

n_loaded_b = 0

for i_dye, dye in enumerate(dyes):
    dye_str = dye.replace("/", "-")

    # Discover all ground-truth files for this dye and parse z/d from the filename
    gt_all = sorted(
        glob.glob(str(save_folder / f"defocus_z*nm_d*nm_*{dye_str}*groundtruth*.csv"))
    )

    for gt_path in gt_all:
        fname = Path(gt_path).name
        m = _flag_re.search(fname)
        if not m:
            continue
        z_val = _parse_nm(m.group(1))
        d_val = _parse_nm(m.group(2))
        i_z   = _nearest_idx(z_offsets_nm,       z_val, tol=1.0)
        i_d   = _nearest_idx(coverslip_depths_nm, d_val, tol=10.0)
        if i_z is None or i_d is None:
            continue

        # Reconstruct flag prefix from the parsed values to find companion files
        flag_prefix = fname[: m.end()]   # everything up to and including the trailing "_"

        ip_files = sorted(glob.glob(str(save_folder / f"{flag_prefix}*{dye_str}*input_parameters*.csv")))
        h5_path  = save_folder / f"{flag_prefix}LM_method_{dye_str}_rawresults.h5"

        if not ip_files or not h5_path.exists():
            continue

        gt    = pd.read_csv(gt_path)
        gt_px = np.vstack([gt["x0"].to_numpy() / pixel_size_nm,
                           gt["y0"].to_numpy() / pixel_size_nm]).T

        dye_BGR     = pd.read_csv(ip_files[0]).to_numpy()[0][-3:]
        colour_true = (dye_BGR / dye_BGR.sum()).reshape(1, 3)

        df_raw = pd.read_hdf(str(h5_path), key="data")

        for i_ph in range(len(n_photon_space)):
            subset = df_raw[df_raw["photon_level"] == i_ph]
            if len(subset) == 0:
                continue

            xc = subset["xc"].to_numpy()
            yc = subset["yc"].to_numpy()
            filt = (
                (xc >= 0) & (yc >= 0) &
                (subset["s_x"].to_numpy() > 0) &
                (subset["s_y"].to_numpy() > 0) &
                (xc <= image_dims) & (yc <= image_dims)
            )
            if filt.sum() < 2:
                continue

            ex = (xc[filt] - gt_px[filt, 0]) * pixel_size_nm
            ey = (yc[filt] - gt_px[filt, 1]) * pixel_size_nm
            bx, by = float(np.nanmean(ex)), float(np.nanmean(ey))
            da_defocus_bias[i_dye, i_z, i_d, i_ph, 0] = np.sqrt((bx**2 + by**2) / 2)

            abgr = np.vstack([
                subset["A_B"].to_numpy()[filt],
                subset["A_G"].to_numpy()[filt],
                subset["A_R"].to_numpy()[filt],
            ]).T
            row_sums = abgr.sum(axis=1, keepdims=True)
            with np.errstate(divide="ignore", invalid="ignore"):
                abgr_norm = np.where(row_sums > 0, abgr / row_sums, np.nan)
            mean_colour = np.nanmean(abgr_norm, axis=0, keepdims=True)
            da_defocus_bias[i_dye, i_z, i_d, i_ph, 1] = float(
                cdist(mean_colour, colour_true)[0, 0]
            )
        n_loaded_b += 1

n_filled_b = int(np.sum(~np.isnan(da_defocus_bias.sel(metric="bias_xy").values)))
print(f"Loaded {n_loaded_b} H5 files → {n_filled_b} grid cells filled "
      f"({n_filled_b / da_defocus_bias.sel(metric='bias_xy').size * 100:.1f} %)")
da_defocus_bias.to_netcdf(str(save_folder / "defocus_bias.nc"))
print(f"bias_xy  : {float(da_defocus_bias.sel(metric='bias_xy').min()):.2f}–"
      f"{float(da_defocus_bias.sel(metric='bias_xy').max()):.2f} nm")
print(f"col. bias: {float(da_defocus_bias.sel(metric='colour_bias').min()):.4f}–"
      f"{float(da_defocus_bias.sel(metric='colour_bias').max()):.4f}")

In [ ]:
# ── Figure 6a: position bias — surface map ────────────────────────────────────
metric     = "bias_xy"
cbar_label = r"Position bias $\sqrt{(\bar{\epsilon}_x^2+\bar{\epsilon}_y^2)/2}$ / nm"

data_m = da_defocus_bias.sel(metric=metric).values
vmin, vmax = 0.0, float(np.nanmax(data_m))
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

fig, axs = plt.subplots(
    nrows=len(coverslip_depths_nm), ncols=len(dyes),
    figsize=(6.69, 2.5 * len(coverslip_depths_nm)),
    squeeze=False,
)
for i_d, d_nm in enumerate(coverslip_depths_nm):
    for i_c, dye in enumerate(dyes):
        ax = axs[i_d, i_c]
        Z = da_defocus_bias.isel(dye=i_c, coverslip_nm=i_d).sel(metric=metric).values
        ax.pcolormesh(z_offsets_nm, n_photon_space, Z.T,
                      cmap="viridis", shading="auto", norm=norm)
        ax.set_yscale("log")
        ax.set_xlabel("Defocus / nm", fontsize=fs)
        ax.set_ylabel(
            f"Photons\n{depth_labels[d_nm]}" if i_c == 0 else "", fontsize=fs - 1,
        )
        if i_d == 0:
            ax.set_title(dye_labels[dye], fontsize=fs)
        ax.tick_params(labelsize=fs - 1)

fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap="viridis"),
    ax=axs, label=cbar_label, shrink=0.7, aspect=30, pad=0.02,
)
plotter.save_or_show(fig, save_path=str(save_folder / "position_bias_surface.svg"))

In [ ]:
# ── Figure 6b: colour bias — surface map ─────────────────────────────────────
metric     = "colour_bias"
cbar_label = r"Colour bias $||\bar{f}_{\mathrm{fit}} - f_{\mathrm{true}}||$"

data_m = da_defocus_bias.sel(metric=metric).values
vmin, vmax = 0.0, float(np.nanmax(data_m))
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

fig, axs = plt.subplots(
    nrows=len(coverslip_depths_nm), ncols=len(dyes),
    figsize=(6.69, 2.5 * len(coverslip_depths_nm)),
    squeeze=False,
)
for i_d, d_nm in enumerate(coverslip_depths_nm):
    for i_c, dye in enumerate(dyes):
        ax = axs[i_d, i_c]
        Z = da_defocus_bias.isel(dye=i_c, coverslip_nm=i_d).sel(metric=metric).values
        ax.pcolormesh(z_offsets_nm, n_photon_space, Z.T,
                      cmap="viridis", shading="auto", norm=norm)
        ax.set_yscale("log")
        ax.set_xlabel("Defocus / nm", fontsize=fs)
        ax.set_ylabel(
            f"Photons\n{depth_labels[d_nm]}" if i_c == 0 else "", fontsize=fs - 1,
        )
        if i_d == 0:
            ax.set_title(dye_labels[dye], fontsize=fs)
        ax.tick_params(labelsize=fs - 1)

fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap="viridis"),
    ax=axs, label=cbar_label, shrink=0.7, aspect=30, pad=0.02,
)
plotter.save_or_show(fig, save_path=str(save_folder / "colour_bias_surface.svg"))

## Comparative Degradation Surfaces

How much worse does each metric get relative to the **in-focus (z = 0 nm) baseline**?

**Precision metrics** (`sigma_xy`, `colour_dist`) are shown as a **fold ratio**
`metric(z) / metric(z = 0)`: value 1 = identical to in-focus, value 2 = twice as bad.

**Bias metrics** (`bias_xy`, `colour_bias`) are shown as the **absolute increase**
`metric(z) − metric(z = 0)`, because the in-focus bias is near zero, making a fold
ratio numerically unstable.

Each figure keeps the 2 rows × 3 cols layout (coverslip depth × dye) matching the
existing surface plots above.  Colourmap: `plasma`; colour scale clipped at the
99th percentile.

In [ ]:
import xarray as xr
import matplotlib.colors as mcolors

# ── Load precision and bias arrays from disk ──────────────────────────────────
da_defocus      = xr.open_dataarray(str(save_folder / "defocus_sweep.nc"))
da_defocus_bias = xr.open_dataarray(str(save_folder / "defocus_bias.nc"))
print(f"Precision : {da_defocus.dims}  {da_defocus.shape}")
print(f"Bias      : {da_defocus_bias.dims}  {da_defocus_bias.shape}")

In [ ]:
# ── Figure 8: precision metrics — fold degradation vs z = 0 nm baseline ──────
# baseline shape: (dye, coverslip_nm, n_photons, metric); broadcasts over z_nm.
baseline_p    = da_defocus.isel(z_nm=0)
ratio_defocus = da_defocus / baseline_p.where(baseline_p > 1e-10)

metric_list_p = ["sigma_xy",                      "colour_dist"]
cbar_lbls_p   = [r"$\sigma_{xy}$ fold × in-focus",
                 r"$\sigma_{\mathrm{colour}}$ fold × in-focus"]
save_names_p  = ["sigma_xy_fold_degradation_surface.svg",
                 "colour_dist_fold_degradation_surface.svg"]

_z = da_defocus.coords["z_nm"].values
_ph = da_defocus.coords["n_photons"].values

for metric, cbar_lbl, save_name in zip(metric_list_p, cbar_lbls_p, save_names_p):
    data_m = ratio_defocus.sel(metric=metric).values   # (n_dye, n_z, n_d, n_ph)
    vmin   = 1.0
    vmax   = float(np.nanpercentile(data_m, 99))
    norm   = mcolors.Normalize(vmin=vmin, vmax=vmax)

    fig, axs = plt.subplots(
        nrows=len(coverslip_depths_nm), ncols=len(dyes),
        figsize=(6.69, 2.5 * len(coverslip_depths_nm)),
        squeeze=False,
    )
    for i_d, d_nm in enumerate(coverslip_depths_nm):
        for i_c, dye in enumerate(dyes):
            ax = axs[i_d, i_c]
            Z  = ratio_defocus.isel(dye=i_c, coverslip_nm=i_d).sel(metric=metric).values
            ax.pcolormesh(_z, _ph, Z.T, cmap="plasma", shading="auto", norm=norm)
            ax.set_yscale("log")
            ax.set_xlabel("Defocus / nm", fontsize=fs)
            ax.set_ylabel(
                f"Photons\n{depth_labels[d_nm]}" if i_c == 0 else "", fontsize=fs - 1,
            )
            if i_d == 0:
                ax.set_title(dye_labels[dye], fontsize=fs)
            ax.tick_params(labelsize=fs - 1)

    fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap="plasma"),
        ax=axs, label=cbar_lbl, shrink=0.7, aspect=30, pad=0.02,
    )
    fig.tight_layout(pad=0.4)
    plotter.save_or_show(fig, save_path=str(save_folder / save_name))

In [ ]:
# ── Figure 9: bias metrics — absolute increase vs z = 0 nm baseline ──────────
baseline_b   = da_defocus_bias.isel(z_nm=0)
increase_b   = da_defocus_bias - baseline_b

metric_list_b = ["bias_xy",                        "colour_bias"]
cbar_lbls_b   = [r"Extra pos. bias / nm",
                 r"Extra colour bias $\Delta||\bar{f}_{\mathrm{fit}} - f_{\mathrm{true}}||$"]
save_names_b  = ["bias_xy_absolute_increase_surface.svg",
                 "colour_bias_absolute_increase_surface.svg"]

_z_b  = da_defocus_bias.coords["z_nm"].values
_ph_b = da_defocus_bias.coords["n_photons"].values

for metric, cbar_lbl, save_name in zip(metric_list_b, cbar_lbls_b, save_names_b):
    data_m = increase_b.sel(metric=metric).values   # (n_dye, n_z, n_d, n_ph)
    vmin   = 0.0
    vmax   = float(np.nanpercentile(data_m, 99))
    norm   = mcolors.Normalize(vmin=vmin, vmax=vmax)

    fig, axs = plt.subplots(
        nrows=len(coverslip_depths_nm), ncols=len(dyes),
        figsize=(6.69, 2.5 * len(coverslip_depths_nm)),
        squeeze=False,
    )
    for i_d, d_nm in enumerate(coverslip_depths_nm):
        for i_c, dye in enumerate(dyes):
            ax = axs[i_d, i_c]
            Z  = increase_b.isel(dye=i_c, coverslip_nm=i_d).sel(metric=metric).values
            ax.pcolormesh(_z_b, _ph_b, Z.T, cmap="plasma", shading="auto", norm=norm)
            ax.set_yscale("log")
            ax.set_xlabel("Defocus / nm", fontsize=fs)
            ax.set_ylabel(
                f"Photons\n{depth_labels[d_nm]}" if i_c == 0 else "", fontsize=fs - 1,
            )
            if i_d == 0:
                ax.set_title(dye_labels[dye], fontsize=fs)
            ax.tick_params(labelsize=fs - 1)

    fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap="plasma"),
        ax=axs, label=cbar_lbl, shrink=0.7, aspect=30, pad=0.02,
    )
    fig.tight_layout(pad=0.4)
    plotter.save_or_show(fig, save_path=str(save_folder / save_name))